In [1]:
%%file producer.py

from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Pięć maszyn w hali produkcyjnej
MACHINES = ['PRASA', 'KOMPRESOR', 'TOKARKA', 'POMPA', 'SILNIK']

# Każda maszyna ma swoje bazowe parametry pracy — lekko różne
random.seed(7)
BASE = {
    m: {
        'temp':     random.uniform(55.0, 68.0),   # °C
        'pressure': random.uniform(4.2,  6.8),    # bar
        'vib':      random.uniform(0.8,  1.9),    # mm/s
    }
    for m in MACHINES
}
random.seed()

# Stan degradacji per maszyna: czy trwa, i który to krok
degradation = {m: {'active': False, 'step': 0} for m in MACHINES}


def generate_reading(machine_id):
    base = BASE[machine_id]
    d = degradation[machine_id]

    # Z prawdopodobieństwem 0.4% startuje degradacja (jeśli żadna nie trwa)
    if not d['active'] and random.random() < 0.004:
        d['active'] = True
        d['step'] = 0
        print(f'\n>>> START DEGRADACJI: {machine_id}\n')

    if d['active']:
        step = d['step']
        # Każdy krok pogarsza parametry maszyny
        temp     = base['temp']     + step * 1.4  + random.gauss(0, 0.5)
        pressure = base['pressure'] - step * 0.11 + random.gauss(0, 0.08)
        vib      = base['vib']      + step * 0.14 + random.gauss(0, 0.07)
        d['step'] += 1
        if d['step'] > 40:
            d['active'] = False
            print(f'\n>>> KONIEC DEGRADACJI: {machine_id} (symulowana naprawa)\n')
    else:
        # Normalna praca: wartości oscylują wokół bazy z małym szumem
        temp     = base['temp']     + random.gauss(0, 1.2)
        pressure = base['pressure'] + random.gauss(0, 0.15)
        vib      = base['vib']      + random.gauss(0, 0.12)

    return {
        'reading_id':  f'R{random.randint(10000,99999)}',
        'machine_id':  machine_id,
        'temperature': round(temp, 2),           # °C
        'pressure':    round(max(0.1, pressure), 3),  # bar
        'vibration':   round(max(0.0, vib), 3),  # mm/s
        'timestamp':   datetime.now().isoformat(),
    }

print('Producent uruchomiony — 5 maszyn, 1 odczyt/s per maszyna\n')
print(f"{'Maszyna':<12} {'Temp':>7} {'Ciśn':>7} {'Drg':>7}  Status")
print('─' * 52)

while True:
    for m in MACHINES:
        reading = generate_reading(m)
        producer.send('machine_raw', value=reading)

        status = 'DEGRADACJA !!!' if degradation[m]['active'] else 'OK'
        print(
            f"{m:<12}"
            f"{reading['temperature']:>6.1f}°C "
            f"{reading['pressure']:>6.2f}bar "
            f"{reading['vibration']:>6.3f}mm/s  {status}"
        )

    producer.flush()
    time.sleep(1.0)

Overwriting producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'machine_raw',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',
    group_id='filter-group',         # własny group_id — niezależny od innych
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Progi bezpośrednich alertów na surowych odczytach
THRESHOLDS = {
    'temperature': {'warn': 75.0,  'critical': 88.0},  # °C
    'pressure':    {'warn': 2.5,   'critical': 1.5},   # bar — niskie = problem
    'vibration':   {'warn': 4.5,   'critical': 6.5},   # mm/s
}

print('Filtr surowych odczytów uruchomiony\n')

for message in consumer:
    r = message.value
    alerts = []

    if r['temperature'] > THRESHOLDS['temperature']['critical']:
        alerts.append(f"TEMP CRITICAL ({r['temperature']}°C)")
    elif r['temperature'] > THRESHOLDS['temperature']['warn']:
        alerts.append(f"TEMP WARN ({r['temperature']}°C)")

    if r['pressure'] < THRESHOLDS['pressure']['critical']:
        alerts.append(f"CIŚN CRITICAL ({r['pressure']}bar)")
    elif r['pressure'] < THRESHOLDS['pressure']['warn']:
        alerts.append(f"CIŚN WARN ({r['pressure']}bar)")

    if r['vibration'] > THRESHOLDS['vibration']['critical']:
        alerts.append(f"DRG CRITICAL ({r['vibration']}mm/s)")
    elif r['vibration'] > THRESHOLDS['vibration']['warn']:
        alerts.append(f"DRG WARN ({r['vibration']}mm/s)")

    if alerts:
        level = 'CRITICAL' if any('CRITICAL' in a for a in alerts) else 'WARN'
        print(f"[{level}] {r['machine_id']:12} | " + ' | '.join(alerts))

Overwriting consumer_filter.py


In [3]:
%%file consumer_aggregator.py
from kafka import KafkaConsumer, KafkaProducer
from collections import defaultdict, deque
import statistics
import json

consumer = KafkaConsumer(
    'machine_raw',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',
    group_id='aggregator-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

agg_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

WINDOW = 15   # rozmiar okna kroczącego

def empty_window():
    # Osobne kolejki dla każdego sensora
    return {
        'temperature': deque(maxlen=WINDOW),
        'pressure':    deque(maxlen=WINDOW),
        'vibration':   deque(maxlen=WINDOW),
    }

# Okna kroczące per maszyna
windows = defaultdict(empty_window)

def compute_slope(values):
    """
    Trend: porównuje średnią drugiej połowy okna z pierwszą.
    Wynik dodatni = wartość rośnie. Wynik ujemny = wartość spada.
    Zwraca zmianę na jeden odczyt.
    """
    if len(values) < WINDOW:
        return 0.0
    lst = list(values)
    mid = len(lst) // 2
    return (statistics.mean(lst[mid:]) - statistics.mean(lst[:mid])) / mid


def compute_zscore(values, current):
    """
    O ile odchyleń standardowych bieżący odczyt różni się od
    średniej okna. Wysoki z-score = nagły skok względem własnej normy.
    """
    if len(values) < 5:
        return 0.0
    mu = statistics.mean(values)
    try:
        sigma = statistics.stdev(values)
    except statistics.StatisticsError:
        return 0.0
    return 0.0 if sigma < 0.001 else (current - mu) / sigma


print(f'Agregator uruchomiony — okno {WINDOW} odczytów per maszyna')
print(f"{'Maszyna':<12} {'n':>3} {'temp_slope':>11} {'pres_slope':>11} {'vib_z':>7}")
print('─' * 52)

for message in consumer:
    r = message.value
    mid = r['machine_id']
    w = windows[mid]

    # Dołącz bieżący odczyt do okien kroczących
    w['temperature'].append(r['temperature'])
    w['pressure'].append(r['pressure'])
    w['vibration'].append(r['vibration'])

    n = len(w['temperature'])

    # Oblicz cechy trendowe
    temp_slope  = compute_slope(w['temperature'])
    pres_slope  = compute_slope(w['pressure'])    # ujemny = ciśnienie spada
    vib_zscore  = compute_zscore(w['vibration'], r['vibration'])

    aggregated = {
        'machine_id':    mid,
        'timestamp':     r['timestamp'],
        'n_readings':    n,
        # Cechy trendowe — wejście dla reguł i modelu ML
        'temp_slope':    round(temp_slope, 4),
        'pres_slope':    round(pres_slope, 4),
        'vib_zscore':    round(vib_zscore, 4),
        # Średnie z okna — do wyświetlania w alertach
        'temp_avg':      round(statistics.mean(w['temperature']), 2),
        'pressure_avg':  round(statistics.mean(w['pressure']), 3),
        'vib_avg':       round(statistics.mean(w['vibration']), 3),
        # Bieżące wartości surowe
        'current_temp':     r['temperature'],
        'current_pressure': r['pressure'],
        'current_vib':      r['vibration'],
    }

    agg_producer.send('machine_aggregated', value=aggregated)

    print(
        f"{mid:<12} {n:>3} "
        f"{temp_slope:>+10.3f} "
        f"{pres_slope:>+10.3f} "
        f"{vib_zscore:>+6.2f}σ"
    )

agg_producer.flush()

Overwriting consumer_aggregator.py


In [4]:
%%file consumer_rules.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'machine_aggregated',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',
    group_id='rules-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

ALERT_THRESHOLD = 4   # minimalna liczba punktów żeby wysłać alert

def score_machine(agg):
    score = 0
    rules = []

    # R1: temperatura wyraźnie rośnie (trend)
    ts = agg['temp_slope']
    if ts > 1.2:
        score += 3
        rules.append(f'R1:temp_rosnie({ts:+.2f}°C/odczyt)')
    elif ts > 0.6:
        score += 1
        rules.append(f'R1b:temp_lekko_rosnie({ts:+.2f}°C/odczyt)')

    # R2: ciśnienie wyraźnie spada (trend ujemny)
    ps = agg['pres_slope']
    if ps < -0.08:
        score += 3
        rules.append(f'R2:cisn_spada({ps:+.3f}bar/odczyt)')
    elif ps < -0.04:
        score += 1
        rules.append(f'R2b:cisn_lekko_spada({ps:+.3f}bar/odczyt)')

    # R3: nagły skok drgań ponad normę maszyny
    vz = agg['vib_zscore']
    if vz > 2.5:
        score += 3
        rules.append(f'R3:drg_skok({vz:.2f}σ)')
    elif vz > 1.8:
        score += 1
        rules.append(f'R3b:drg_podwyzszone({vz:.2f}σ)')

    # R4: temperatura bezwzględnie wysoka (bez względu na trend)
    if agg['temp_avg'] > 87.0:
        score += 2
        rules.append(f"R4:temp_wysoka({agg['temp_avg']:.1f}°C)")

    return score, rules


print(f'Konsument regułowy uruchomiony (próg alertu: {ALERT_THRESHOLD} pkt)\n')

for message in consumer:
    agg = message.value

    # Ignoruj pierwsze odczyty — okno nie jest jeszcze wypełnione
    if agg['n_readings'] < 10:
        continue

    score, rules = score_machine(agg)

    if score >= ALERT_THRESHOLD:
        alert = {
            **agg,
            'score':        score,
            'rules':        rules,
            'alert_source': 'rules',
        }
        alert_producer.send('alerts', value=alert)
        alert_producer.flush()
        print(
            f"[ALERT {score:2d}pkt] {agg['machine_id']:12} | "
            + ' | '.join(rules)
        )
    else:
        print(f"[OK    {score:2d}pkt] {agg['machine_id']:12}")

Overwriting consumer_rules.py


In [5]:
%%file train_model.py
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

N_NORMAL = 3000
N_FAIL   = 200   # ~6% — tyle degradacji pojawi się w strumieniu

# Normalna praca: trendy bliskie zera, brak skoków
normal = pd.DataFrame({
    'temp_slope':  np.random.normal(0.0,  0.28, N_NORMAL).clip(-0.9, 0.9),
    'pres_slope':  np.random.normal(0.0,  0.03, N_NORMAL).clip(-0.1, 0.1),
    'vib_zscore':  np.random.normal(0.0,  0.7,  N_NORMAL).clip(-2.2, 2.2),
    'label': 0
})

# Degradacja: temperatura rośnie, ciśnienie spada, drgania skaczą
failing = pd.DataFrame({
    'temp_slope':  np.random.normal(2.1,  0.55, N_FAIL).clip(1.1, 5.0),
    'pres_slope':  np.random.normal(-0.14, 0.04, N_FAIL).clip(-0.4, -0.05),
    'vib_zscore':  np.random.normal(3.2,  0.8,  N_FAIL).clip(1.5, 6.5),
    'label': 1
})

df = pd.concat([normal, failing], ignore_index=True).sample(frac=1, random_state=42)
FEATURES = ['temp_slope', 'pres_slope', 'vib_zscore']
X, y = df[FEATURES], df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('=' * 55)
print('RANDOM FOREST — wyniki na zbiorze testowym')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['normalna', 'awaria']))

print('Ważność cech:')
for feat, imp in sorted(zip(FEATURES, clf.feature_importances_), key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'  {feat:<14} {bar} {imp:.3f}')

with open('machine_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

print('\nModel zapisany: machine_model.pkl')

Overwriting train_model.py


In [6]:
%%file machine_api.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pickle, numpy as np

app = FastAPI(title='Machine Anomaly Detection API')

model = pickle.load(open('machine_model.pkl', 'rb'))

FEATURES = ['temp_slope', 'pres_slope', 'vib_zscore']


class MachineFeatures(BaseModel):
    temp_slope:  float = Field(..., example=0.12,  description='Trend temperatury [°C/odczyt]')
    pres_slope:  float = Field(..., example=-0.02, description='Trend ciśnienia [bar/odczyt]')
    vib_zscore:  float = Field(..., example=0.8,   description='Skok drgań względem normy [σ]')


class PredictionResponse(BaseModel):
    failure_probability: float
    health_score:        int
    risk_level:          str
    is_anomaly:          bool


def risk_label(prob):
    if prob >= 0.75: return 'CRITICAL'
    if prob >= 0.50: return 'HIGH'
    if prob >= 0.25: return 'MEDIUM'
    return 'LOW'


@app.post('/score', response_model=PredictionResponse)
def score(data: MachineFeatures):
    X = np.array([[data.temp_slope, data.pres_slope, data.vib_zscore]])
    prob   = float(model.predict_proba(X)[0, 1])   # prawdop. klasy 1 (awaria)
    health = max(0, min(100, int((1.0 - prob) * 100)))
    return {
        'failure_probability': round(prob, 4),
        'health_score':        health,
        'risk_level':          risk_label(prob),
        'is_anomaly':          bool(prob >= 0.5),
    }


@app.get('/health')
def health():
    return {'status': 'ok'}


@app.get('/model-info')
def model_info():
    return {
        'type':     'RandomForestClassifier',
        'features': FEATURES,
        'n_estimators': model.n_estimators,
        'output': {
            'failure_probability': '0.0 – 1.0',
            'health_score':        '0 (awaria) – 100 (ideał)',
            'risk_level':          'LOW | MEDIUM | HIGH | CRITICAL',
        }
    }

Overwriting machine_api.py


In [7]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

consumer = KafkaConsumer(
    'machine_aggregated',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',
    group_id='ml-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = 'http://localhost:8001/score'
ALERT_PROB_THRESHOLD = 0.50

print('Konsument ML uruchomiony — czyta z machine_aggregated\n')
print(f"{'Maszyna':<12} {'Health':>6} {'P(awaria)':>10} {'Risk'}")
print('─' * 42)

for message in consumer:
    agg = message.value

    if agg['n_readings'] < 10:
        continue

    payload = {
        'temp_slope':  agg['temp_slope'],
        'pres_slope':  agg['pres_slope'],
        'vib_zscore':  agg['vib_zscore'],
    }

    try:
        resp   = requests.post(API_URL, json=payload, timeout=2)
        result = resp.json()
    except requests.RequestException as e:
        print(f'[!] API niedostępne: {e}')
        continue

    prob   = result['failure_probability']
    health = result['health_score']
    risk   = result['risk_level']

    if result['is_anomaly'] or prob >= ALERT_PROB_THRESHOLD:
        alert = {
            **agg,
            'failure_probability': prob,
            'health_score':        health,
            'risk_level':          risk,
            'alert_source':        'ml_model',
        }
        alert_producer.send('alerts', value=alert)
        alert_producer.flush()

    # Kolorystyka w terminalu wg poziomu ryzyka
    if risk == 'CRITICAL':
        line = f'>>> AWARIA   P={prob:.0%}'
    elif risk == 'HIGH':
        line = f'!   WYSOKIE  P={prob:.0%}'
    elif risk == 'MEDIUM':
        line = f'~   UWAGA    P={prob:.0%}'
    else:
        line = '    OK'

    print(f'{agg["machine_id"]:<12} {health:>6}  {prob:>9.1%}  {risk:<9}  {line}')

Overwriting ml_consumer.py
